In [2]:
import sys
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from IPython.utils import io
import hashlib

import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.collate_batch import collate_batch
from src.answer_set import AnswerSet

# from src.data_preprocessing import save_assistments_DKT
# save_assistments_DKT()
from models.model_dkt import DKT


In [5]:
with open('data/preprocessed/assistments_user_dict.json', 'r') as json_file:
    user_dict = json.load(json_file)

The number of unique exercises is too high, we need random vector representations.
We should transform to the following dimension:

In [17]:
round(np.log(dim))

10

In [7]:


# Now create your dataset and dataloader
dataset = AnswerSet(user_dict)
loader = DataLoader(dataset, batch_size=100, collate_fn=collate_batch, num_workers=0)  # Set num_workers to 0 for testing
# just testing the output
sequences, labels, lengths = next(iter(loader))
print(f"Sequences shape: {sequences.shape}")
print(f"Labels shape: {labels.shape}")
print(f"Lengths: {lengths}")


Sequences shape: torch.Size([100, 820, 2])
Labels shape: torch.Size([100, 820])
Lengths: tensor([ 24,  21,   5,   2, 303,  23,   9, 615,   9,   9,  31,   9, 116,   1,
         20, 290,   5,  47, 276, 117, 187, 154, 113, 194,  25,  84, 314, 259,
        235, 467, 325, 152, 651, 514,  78, 721, 144, 680, 515, 363, 410, 680,
         92, 312, 384,  81, 366, 381, 820, 128,  28,  88,  34, 482,  88, 178,
        508, 271, 476,  60,  55,   9,   1,  65,   2,  14,   9,  13,  20,   9,
         12,   9,   7,  31,  38, 152,   9,  13,  33,   2,  17,  10,  35,  49,
          7,  32,  20,  32,  21,  14,  48,  33,  45,  19,   9,   9,   9,  20,
         93,  53])


In [8]:
# Instantiate the model
num_items = 300  # Total number of different questions
embed_dim = 10   # Embedding dimension from transform_to_random_vector
hid_size = 200    # Hidden layer size in the RNN
num_hid_layers = 2  # Number of hidden layers in the RNN
drop_prob = 0.5     # Dropout probability

model = DKT(num_items=num_items, embed_dim=embed_dim, hid_size=hid_size, num_hid_layers=num_hid_layers, drop_prob=drop_prob)

# Forward pass
output = model(sequences, lengths)
print(output)


tensor([[[0.4856, 0.4718, 0.5369,  ..., 0.5280, 0.5060, 0.4933],
         [0.4486, 0.4891, 0.5224,  ..., 0.5234, 0.5060, 0.4600],
         [0.4796, 0.4498, 0.5441,  ..., 0.5199, 0.4934, 0.4644],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.4838, 0.5081, 0.5629,  ..., 0.5332, 0.5106, 0.5229],
         [0.4761, 0.4654, 0.5065,  ..., 0.5162, 0.4923, 0.4844],
         [0.4623, 0.4949, 0.5422,  ..., 0.5145, 0.4897, 0.4326],
         ...,
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.0000,  ..., 0.0000, 0.0000, 0.0000]],

        [[0.4863, 0.5061, 0.5332,  ..., 0.5240, 0.4984, 0.5132],
         [0.4978, 0.4862, 0.5400,  ..., 0.5310, 0.5036, 0.4834],
         [0.5177, 0.5115, 0.5306,  ..., 0.5239, 0.4765, 0.

In [ ]:
loss = F.binary_cross_entropy(masked_output, target, reduction='none')
loss = loss * mask  # Zero out the padded positions in the loss
loss = loss.sum() / mask.sum()  # Average only over valid positions